## read the result batch sampling files

In [1]:
import h5py
import numpy as np
from pathlib import Path

DATASET_DIR = Path("./dataset_v3")
SAMPLING_ID = 2
RUNS_PER_BATCH = 8000
RESULT_PATHS = sorted(DATASET_DIR.glob(f"results_batch_*_sampling_{SAMPLING_ID}.hdf5"))
if not RESULT_PATHS:
    raise FileNotFoundError(f"No result shards found in {DATASET_DIR} for sampling_{SAMPLING_ID}")
AVAILABLE_BATCH_IDS = [int(path.name.split("_batch_")[1].split("_")[0]) for path in RESULT_PATHS]
LABELS_PATH = DATASET_DIR / f"completed_systems_sampling_{SAMPLING_ID}.txt"
ICS_PATH = DATASET_DIR / f"ICs_sampling_{SAMPLING_ID}.hdf5"
INITIAL_KEYS = ["aratios", "mratios", "eccs", "incs", "APs", "LANs"]

print("Loading results shards from", DATASET_DIR)
print("Available result batches:", AVAILABLE_BATCH_IDS)
print("Inspecting:", RESULT_PATHS[0])
with h5py.File(RESULT_PATHS[0], "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP] {name}")
    f.visititems(show)

KEYS = [
    "APs_complete", "LANs_complete", "angMomentum_complete", "aratios_complete",
    "eccs_complete", "energy_complete", "incs_complete", "masses_complete",
    "mratios_complete", "periods_complete", "positions_complete",
    "smaxes_complete", "tAnos_complete", "times_complete", "velocities_complete"
]

def load_one_h5(path, T=None, dtype=np.float32):
    sl = slice(None) if T is None else slice(0, T)
    out = {}
    with h5py.File(path, "r") as f:
        for k in KEYS:
            if k == "times_complete":
                out[k] = f[k][:, sl].astype(dtype)
            else:
                out[k] = f[k][:, sl, ...].astype(dtype)
    return out

def load_initial_conditions(path, batch_ids, rows_per_batch=RUNS_PER_BATCH, dtype=np.float32):
    out = {k: [] for k in INITIAL_KEYS}
    with h5py.File(path, "r") as f:
        for batch_id in batch_ids:
            start = batch_id * rows_per_batch
            stop = start + rows_per_batch
            for k in INITIAL_KEYS:
                out[k].append(f[k][start:stop].astype(dtype))
    return {k: np.concatenate(v, axis=0) for k, v in out.items()}

# Keep the original early-prefix budget by default; increase to 200 or 500 for a longer v3 prefix.
T = 100

dicts = [load_one_h5(p, T=T) for p in RESULT_PATHS]
data = {k: np.concatenate([d[k] for d in dicts], axis=0) for k in KEYS}
initial_data = load_initial_conditions(ICS_PATH, AVAILABLE_BATCH_IDS)
print("data load complete")
print("initial conditions load complete")


Loading results shards from dataset_v3
Available result batches: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Inspecting: dataset_v3/results_batch_00_sampling_2.hdf5
[DATASET] APs_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] LANs_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] angMomentum_complete shape=(8000, 2001, 3) dtype=float64
[DATASET] aratios_complete shape=(8000, 2001, 1) dtype=float64
[DATASET] eccs_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] energy_complete shape=(8000, 2001, 1) dtype=float64
[DATASET] incs_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] masses_complete shape=(8000, 2001, 3) dtype=float64
[DATASET] mratios_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] periods_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] positions_complete shape=(8000, 2001, 3, 3) dtype=float64
[DATASET] smaxes_complete shape=(8000, 2001, 2) dtype=float64
[DATASET] tAnos_complete shape=(8000, 2001, 2) dtype=float6

In [2]:
# sanity-check the concatenated prefix tensors for the available dataset_v3 shards
N = RUNS_PER_BATCH * len(RESULT_PATHS)
assert data["positions_complete"].shape == (N, T, 3, 3)
assert data["velocities_complete"].shape == (N, T, 3, 3)
assert data["times_complete"].shape == (N, T)
assert initial_data["aratios"].shape[0] == N


In [3]:
import pandas as pd
import torch

labels = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None,
                 names=["index", "num_batch", "time_unbound", "wall_time", "diagnosis"])

labels = labels.astype({"index": "int64", "num_batch": "int64", "time_unbound": "float32", "wall_time": "float32", "diagnosis": "int64"})
labels["batch_id"] = labels["index"] // RUNS_PER_BATCH
labels["row_in_batch"] = labels["index"] % RUNS_PER_BATCH

if not (labels["num_batch"].to_numpy() == labels["batch_id"].to_numpy()).all():
    mismatch_count = int((labels["num_batch"] != labels["batch_id"]).sum())
    raise ValueError(f"completed_systems batch column disagrees with run-id-derived batch for {mismatch_count} rows")

labels = labels[labels["batch_id"].isin(AVAILABLE_BATCH_IDS)].copy()
labels = labels.sort_values(["batch_id", "row_in_batch"]).reset_index(drop=True)

arr = labels.to_numpy()
print(labels.columns)
print(arr.shape, arr.dtype)
print(arr[:10, :7])
assert len(labels) == data["times_complete"].shape[0]


Index(['index', 'num_batch', 'time_unbound', 'wall_time', 'diagnosis',
       'batch_id', 'row_in_batch'],
      dtype='object')
(160000, 7) float64
[[0.00000000e+00 0.00000000e+00 1.00000000e+04 1.28617844e+02
  0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00 1.00000000e+04 1.26132133e+02
  0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [2.00000000e+00 0.00000000e+00 1.00000000e+04 1.27196609e+02
  0.00000000e+00 0.00000000e+00 2.00000000e+00]
 [3.00000000e+00 0.00000000e+00 1.00000000e+04 1.27244606e+02
  0.00000000e+00 0.00000000e+00 3.00000000e+00]
 [4.00000000e+00 0.00000000e+00 1.00000000e+04 1.29052979e+02
  0.00000000e+00 0.00000000e+00 4.00000000e+00]
 [5.00000000e+00 0.00000000e+00 1.00000000e+04 1.28769531e+02
  0.00000000e+00 0.00000000e+00 5.00000000e+00]
 [6.00000000e+00 0.00000000e+00 1.00000000e+04 1.27129807e+02
  0.00000000e+00 0.00000000e+00 6.00000000e+00]
 [7.00000000e+00 0.00000000e+00 1.00000000e+04 1.26864037e+02
  0.00000000e+00 0.

In [4]:
def nan_report(data, T, max_show=5):
    rep = []
    N = next(iter(data.values())).shape[0]
    for k, a in data.items():
        b = a[:, :T, ...] if (a.ndim >= 2 and a.shape[1] >= T) else a

        nan_mask = ~np.isfinite(b)
        bad = nan_mask.any(axis=tuple(range(1, b.ndim)))
        nan_count = int(bad.sum())

        # collect a few NaN coordinates like (sample, t, ...)
        pos = np.argwhere(nan_mask)
        examples = [tuple(map(int, p)) for p in pos[:max_show]]

        rep.append((k, nan_count, N, examples))

    return sorted(rep, key=lambda x: x[1], reverse=True)

for k, bad, N, examples in nan_report(data, T, max_show=5):
    print(k, bad, "/", N)
    if examples:
        print("  NaN examples (sample, time, ...):", examples)


periods_complete 65989 / 160000
  NaN examples (sample, time, ...): [(40, 73, 0), (40, 74, 0), (40, 75, 0), (40, 76, 0), (40, 76, 1)]
tAnos_complete 36157 / 160000
  NaN examples (sample, time, ...): [(0, 0, 0), (0, 0, 1), (4, 0, 1), (8, 0, 1), (10, 0, 1)]
APs_complete 0 / 160000
LANs_complete 0 / 160000
angMomentum_complete 0 / 160000
aratios_complete 0 / 160000
eccs_complete 0 / 160000
energy_complete 0 / 160000
incs_complete 0 / 160000
masses_complete 0 / 160000
mratios_complete 0 / 160000
positions_complete 0 / 160000
smaxes_complete 0 / 160000
times_complete 0 / 160000
velocities_complete 0 / 160000


In [5]:
print(data['tAnos_complete'][6])

[[1.4585451  3.1415927 ]
 [3.0867577  3.139361  ]
 [2.7044954  3.141212  ]
 [3.3493845  3.1435027 ]
 [3.3651085  3.1464605 ]
 [3.6184425  3.1466339 ]
 [6.252858   3.1511185 ]
 [2.517469   3.1536705 ]
 [3.1073072  3.1559258 ]
 [3.0493054  3.1579893 ]
 [3.861449   3.161287  ]
 [3.5114162  3.1650944 ]
 [4.6026134  3.1662366 ]
 [1.5799243  3.1683788 ]
 [2.9177938  3.169275  ]
 [3.1362     3.1730127 ]
 [3.4688098  3.1701496 ]
 [4.730541   3.1740353 ]
 [2.3951101  3.173498  ]
 [2.360699   3.1753516 ]
 [2.5664933  3.174928  ]
 [3.2219357  3.178672  ]
 [3.2253482  3.1767025 ]
 [4.173466   3.1779854 ]
 [1.0514545  3.1790097 ]
 [2.9718437  3.1799314 ]
 [2.8079307  3.180271  ]
 [3.2068326  3.1847608 ]
 [3.4989133  3.1861246 ]
 [3.391379   3.1886916 ]
 [5.7874436  3.1890595 ]
 [2.3097124  3.1913955 ]
 [3.091092   3.193721  ]
 [3.0692408  3.1952143 ]
 [3.8352635  3.1959848 ]
 [3.9002662  3.1954975 ]
 [4.687861   3.1976867 ]
 [1.6383185  3.1986456 ]
 [2.8747985  3.2027874 ]
 [3.1465123  3.201759  ]


### A) Phase Variations (`sampling_2`)
- Each unique system appears in a **4 x 4 phase grid**, giving **16 phase variations**.
- These 16 variations are repeated in a second block with a **perturbed inner semimajor axis**, for **32 runs per unique system**.
- Since orbital phases evolve continuously, we still **exclude raw phase variables** from the static ML feature vector.

### B) Sensitivity / Chaos Perturbations
- The second 16-run block applies a **small semimajor-axis perturbation** while reusing the same phase grid.
- This preserves the Task B setup as a prefix-to-outcome problem, while letting `dataset_v3` probe sensitivity to nearby initial conditions.


In [6]:
import numpy as np

# A sample is bad if it has an invalid label or contains NaN/Inf in geometry channels.
valid_label = (labels["time_unbound"] >= 0) & (labels["diagnosis"] == 0)
bad_pos = ~np.isfinite(data["positions_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
bad_vel = ~np.isfinite(data["velocities_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
mask = valid_label.to_numpy() & ~(bad_pos | bad_vel)

data = {k: v[mask] for k, v in data.items()}
initial_data = {k: v[mask] for k, v in initial_data.items()}
labels = labels.loc[mask].reset_index(drop=True)

TIME_HORIZON = float(labels["time_unbound"].max())
NUM_TIME_BINS = 64
SOFT_LABEL_SIGMA_BINS = 1.5

def build_soft_time_targets(
    times: np.ndarray,
    t_max: float = TIME_HORIZON,
    num_bins: int = NUM_TIME_BINS,
    sigma_bins: float = SOFT_LABEL_SIGMA_BINS,
):
    """Convert scalar `time_unbound` into a soft distribution over time bins."""
    times = np.asarray(times, dtype=np.float32)
    clipped = np.clip(times, 0.0, t_max)

    bin_edges = np.linspace(0.0, t_max, num_bins + 1, dtype=np.float32)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_centers[-1] = t_max

    bin_width = float(t_max) / float(num_bins)
    sigma = max(float(sigma_bins) * bin_width, 1e-6)

    dist = np.exp(-0.5 * ((clipped[:, None] - bin_centers[None, :]) / sigma) ** 2).astype(np.float32)

    censored = np.isclose(clipped, t_max, rtol=0, atol=1e-6)
    dist[censored] = 0.0
    dist[censored, -1] = 1.0

    dist /= dist.sum(axis=1, keepdims=True) + 1e-8
    return clipped, dist, bin_edges, bin_centers, censored.astype(np.int32)

time_target, y_soft, TIME_BIN_EDGES, TIME_BIN_CENTERS, y_hard = build_soft_time_targets(
    labels["time_unbound"].to_numpy()
)
y = y_hard

assert len(y_soft) == data["times_complete"].shape[0]
print("After cleaning:", len(y_soft), "censored:", int(y_hard.sum()))
print("TIME_HORIZON:", TIME_HORIZON)
print("y_soft shape:", y_soft.shape)
print("observed unbound:", int((y_hard == 0).sum()), "censored at horizon:", int((y_hard == 1).sum()))


After cleaning: 159953 censored: 50400
TIME_HORIZON: 10000.0
y_soft shape: (159953, 64)
observed unbound: 109553 censored at horizon: 50400


In [7]:
import numpy as np

N = len(y_hard)
test_size = min(max(5000, int(round(0.125 * N))), max(1, N // 2))
rng = np.random.default_rng(42)

# Keep the split roughly stratified by censored vs observed cases.
pos_idx = np.where(y_hard == 1)[0]
neg_idx = np.where(y_hard == 0)[0]

n_pos = int(round(test_size * len(pos_idx) / N))
n_neg = test_size - n_pos

n_pos = min(n_pos, len(pos_idx))
n_neg = min(n_neg, len(neg_idx))
remain = test_size - (n_pos + n_neg)
if remain > 0:
    if len(pos_idx) - n_pos >= remain:
        n_pos += remain
    else:
        n_neg += remain

test_pos = rng.choice(pos_idx, size=n_pos, replace=False)
test_neg = rng.choice(neg_idx, size=n_neg, replace=False)

test_idx = np.concatenate([test_pos, test_neg])
rng.shuffle(test_idx)

train_mask = np.ones(N, dtype=bool)
train_mask[test_idx] = False

train_idx_full = np.where(train_mask)[0]
y_train_full = y_hard[train_mask]
val_frac = 0.10
val_size = max(1, int(round(val_frac * len(train_idx_full))))

pos_train = train_idx_full[y_train_full == 1]
neg_train = train_idx_full[y_train_full == 0]
n_val_pos = min(len(pos_train), int(round(val_size * len(pos_train) / len(train_idx_full))))
n_val_neg = min(len(neg_train), val_size - n_val_pos)
remain = val_size - (n_val_pos + n_val_neg)
if remain > 0:
    if len(pos_train) - n_val_pos >= remain:
        n_val_pos += remain
    else:
        n_val_neg += remain

val_pos = rng.choice(pos_train, size=n_val_pos, replace=False)
val_neg = rng.choice(neg_train, size=n_val_neg, replace=False)
val_idx = np.concatenate([val_pos, val_neg])
rng.shuffle(val_idx)

fit_mask = train_mask.copy()
fit_mask[val_idx] = False

data_train = {k: v[fit_mask] for k, v in data.items()}
data_val   = {k: v[val_idx]  for k, v in data.items()}
data_test  = {k: v[test_idx] for k, v in data.items()}
initial_train = {k: v[fit_mask] for k, v in initial_data.items()}
initial_val   = {k: v[val_idx]  for k, v in initial_data.items()}
initial_test  = {k: v[test_idx] for k, v in initial_data.items()}

labels_train = labels.loc[fit_mask].reset_index(drop=True)
labels_val   = labels.loc[val_idx].reset_index(drop=True)
labels_test  = labels.loc[test_idx].reset_index(drop=True)

y_train = y_hard[fit_mask]
y_val   = y_hard[val_idx]
y_test  = y_hard[test_idx]

time_train = time_target[fit_mask]
time_val   = time_target[val_idx]
time_test  = time_target[test_idx]

TRAINING_CONFIG = {
    "svm": {"epochs": 60, "batch_size": 2048, "lr": 3e-3, "weight_decay": 5e-4, "patience": 8},
    "cnn": {"epochs": 40, "batch_size": 512, "lr": 5e-4, "weight_decay": 1e-4, "patience": 6},
    "tcn": {"epochs": 40, "batch_size": 128, "lr": 7e-4, "weight_decay": 1e-4, "patience": 6},
    "attn": {"epochs": 40, "batch_size": 128, "lr": 3e-4, "weight_decay": 1e-4, "patience": 8},
    "gnn": {"epochs": 30, "batch_size": 192, "lr": 5e-4, "weight_decay": 1e-4, "patience": 6},
}

print("train hard-label stats:", len(y_train), "censored:", int(y_train.sum()))
print("val   hard-label stats:", len(y_val),   "censored:", int(y_val.sum()))
print("test  hard-label stats:", len(y_test),  "censored:", int(y_test.sum()))
print("time range train:", float(time_train.min()), float(time_train.max()))


train hard-label stats: 125963 censored: 39690
val   hard-label stats: 13996 censored: 4410
test  hard-label stats: 19994 censored: 6300
time range train: 5.0 10000.0


## SVM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class LinearSVM(nn.Module):
    """
    Linear SVM classifier (binary).
    Uses hinge loss: max(0, 1 - y * f(x)), with y in {-1, +1}.
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # returns raw score f(x) (shape: [B])
        return self.linear(x).squeeze(-1)

def hinge_loss(scores: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    scores: f(x), shape [B]
    y: labels in {-1, +1}, shape [B]
    """
    return torch.clamp(1.0 - y * scores, min=0.0).mean()

@torch.no_grad()
def predict(scores: torch.Tensor) -> torch.Tensor:
    # return predicted label in {0,1}
    return (scores >= 0).long()

def train_svm(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor | None = None,
    y_val: torch.Tensor | None = None,
    epochs: int = 60,
    batch_size: int = 2048,
    lr: float = 3e-3,
    weight_decay: float = 1e-4,  # L2 regularization strength (SVM style)
    patience: int = 8,
    device: str | torch.device = "cpu",
):
    """
    y_train can be {0,1} or {-1,+1}.
    """
    device = torch.device(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    # Convert labels to {-1, +1}
    if y_train.min() >= 0:
        y_train_svm = y_train * 2 - 1
    else:
        y_train_svm = y_train
    y_train_svm = y_train_svm.float()

    in_dim = X_train.shape[1]
    model = LinearSVM(in_dim).to(device)

    ds = TensorDataset(X_train, y_train_svm)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    def l2_penalty(m: nn.Module) -> torch.Tensor:
        w = m.linear.weight
        return (w * w).sum()

    best_acc = -1.0
    best_state = None
    stale_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        nb = 0

        for xb, yb in dl:
            scores = model(xb)
            loss = hinge_loss(scores, yb) + 0.5 * weight_decay * l2_penalty(model)

            opt.zero_grad(set_to_none=True)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # NEW

            opt.step()
            total += loss.item()
            nb += 1

        msg = f"epoch {ep:03d} | train_loss {(total/nb):.6f}"

        # optional validation
        if X_val is not None and y_val is not None:
            model.eval()
            Xv = X_val.to(device)
            yv = y_val.to(device)
            yv01 = yv.clone()
            if yv01.min() < 0:  # {-1,+1} -> {0,1}
                yv01 = (yv01 > 0).long()
            scores_v = model(Xv)
            pred_v = predict(scores_v)
            acc = (pred_v == yv01.long()).float().mean().item()
            msg += f" | val_acc {acc:.4f}"
            if acc > best_acc + 1e-4:
                best_acc = acc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1

        print(msg)
        if X_val is not None and y_val is not None and stale_epochs >= patience:
            print(f"early stopping at epoch {ep:03d} | best_val_acc {best_acc:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


@torch.no_grad()
def accuracy(model, X, y, device="cpu"):
    model.eval()
    X = X.to(device)
    y = y.to(device)

    # y can be {0,1} or {-1,+1}; convert to {0,1} for acc
    if y.min() < 0:
        y01 = (y > 0).long()
    else:
        y01 = y.long()

    scores = model(X)                 # raw score f(x)
    pred = (scores >= 0).long()       # decision boundary at 0
    acc = (pred == y01).float().mean().item()
    return acc

In [ ]:
X_train = build_features(data_train, T=T, add_time=True, use_geom=True)
X_test  = build_features(data_test,  T=T, add_time=True, use_geom=True)

print("X_train:", X_train.shape, "finite:", np.isfinite(X_train).all())
print("X_test :", X_test.shape,  "finite:", np.isfinite(X_test).all())

# flatten
X_train_2d = torch.from_numpy(X_train).float().reshape(X_train.shape[0], -1)
X_val_2d   = torch.from_numpy(build_features(data_val, T=T, add_time=True, use_geom=True)).float().reshape(len(y_val), -1)
X_test_2d  = torch.from_numpy(X_test ).float().reshape(X_test.shape[0], -1)

y_train_t = torch.from_numpy(y_train).long()
y_val_t   = torch.from_numpy(y_val).long()
y_test_t  = torch.from_numpy(y_test ).long()

eps = 1e-8
mu = X_train_2d.mean(dim=0, keepdim=True)
std = X_train_2d.std(dim=0, keepdim=True).clamp_min(eps)

X_train_2d = (X_train_2d - mu) / std
X_val_2d   = (X_val_2d   - mu) / std
X_test_2d  = (X_test_2d  - mu) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = train_svm(
    X_train=X_train_2d,
    y_train=y_train_t,
    X_val=X_val_2d,
    y_val=y_val_t,
    epochs=TRAINING_CONFIG["svm"]["epochs"],
    batch_size=TRAINING_CONFIG["svm"]["batch_size"],
    lr=TRAINING_CONFIG["svm"]["lr"],
    weight_decay=TRAINING_CONFIG["svm"]["weight_decay"],
    patience=TRAINING_CONFIG["svm"]["patience"],
    device=device
)

train_acc = accuracy(model, X_train_2d, y_train_t, device=device)
test_acc  = accuracy(model, X_test_2d,  y_test_t,  device=device)

print("train_acc:", train_acc)
print("test_acc :", test_acc)


## 1D CNN

In [8]:
"""
End-to-end: build (B, C, T) features from your dataset + train a 1D-CNN
for hard-label binary classification using the first T=100 steps.

Features included (per time step):
A) Precomputed orbital/physical series (with angle sin/cos):
  - aratios (1)
  - mratios (2)
  - masses (3)
  - eccs (2)
  - smaxes (2)
  - energy (1)
  - angMomentum (3)
  - APs (2) -> sin/cos (4)
  - LANs (2) -> sin/cos (4)
  - incs (2) -> sin/cos (4)

B) Geometry from positions/velocities (translation/rotation invariant):
  - pairwise distances: r12, r13, r23 (3)
  - pairwise relative speed magnitudes: |v12|, |v13|, |v23| (3)
  - pairwise radial speeds: rdot12, rdot13, rdot23 (3)

C) Optional: normalized time channel (1)

Total channels ~ 38 (A+B+time; periods/tAnos excluded by design).
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# ---------------------------
# Feature engineering helpers
# ---------------------------

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    """
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)

def build_features(data: dict, T: int = 200, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)
    en = data["energy_complete"][:, sl, :].astype(np.float32)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)

    # Angles -> sin/cos  (not use tAnos)
    ap = sincos(data["APs_complete"][:, sl, :])
    lan = sincos(data["LANs_complete"][:, sl, :])
    inc = sincos(data["incs_complete"][:, sl, :])

    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)
        geom = pairwise_geom(pos, vel)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]
        t = np.repeat(t, N, axis=0)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)  # (N,T,C)
    X = X.transpose(0, 2, 1)            # (N,C,T)
    return X.astype(np.float32)

def standardize_train_only(X_tr: np.ndarray, X_te: np.ndarray, eps: float = 1e-6):
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + eps
    X_tr_n = (X_tr - mu) / sd
    X_te_n = (X_te - mu) / sd
    return X_tr_n, X_te_n, (mu, sd)

# ---------------------------
# Dataset / Model
# ---------------------------

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]

class CNN1DClassifier(nn.Module):
    """1D CNN for hard-label binary classification."""
    def __init__(self, in_ch: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(in_ch, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

# ---------------------------
# Training / Evaluation
# ---------------------------

@torch.no_grad()
def eval_cnn_classifier(model, loader, device):
    model.eval()
    all_prob, all_target = [], []

    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).squeeze(-1)
        prob = torch.sigmoid(logits)
        all_prob.append(prob.detach().cpu().numpy())
        all_target.append(yb.numpy())

    all_prob = np.concatenate(all_prob, axis=0)
    all_target = np.concatenate(all_target, axis=0)
    pred = (all_prob >= 0.5).astype(np.int32)

    metrics = {
        "auc": float(roc_auc_score(all_target, all_prob)),
        "f1": float(f1_score(all_target, pred)),
        "accuracy": float(accuracy_score(all_target, pred)),
    }
    return metrics, all_prob, pred, all_target

def train_cnn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    data_val: dict | None = None,
    y_val: np.ndarray | None = None,
    T: int = 100,
    epochs: int = 40,
    batch_size: int = 512,
    lr: float = 5e-4,
    add_time: bool = True,
    use_geom: bool = True,
    weight_decay: float = 1e-4,
    patience: int = 6,
):
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0]

    if (data_val is not None) and (y_val is not None):
        X_va = build_features(data_val, T=T, add_time=add_time, use_geom=use_geom)
        y_va = np.asarray(y_val).astype(np.float32)
        X_tr, X_va, norm = standardize_train_only(X_tr, X_va)
        val_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=1024, shuffle=False)
    else:
        mu = X_tr.mean(axis=(0, 2), keepdims=True)
        sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
        X_tr = (X_tr - mu) / sd
        norm = (mu, sd)
        val_loader = None

    train_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN1DClassifier(in_ch=X_tr.shape[1]).to(device)

    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=lr * 0.1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_auc = -1.0
    best_state = None
    stale_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_acc = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(xb).squeeze(-1)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_acc += float((pred == yb).float().mean().item()) * bs
            total_n += bs

        msg = (
            f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f} "
            f"| train_acc {total_acc/total_n:.4f}"
        )

        if val_loader is not None:
            val_metrics, _, _, _ = eval_cnn_classifier(model, val_loader, device)
            val_auc = val_metrics["auc"]
            msg += f" | val_auc {val_auc:.4f} | val_f1 {val_metrics['f1']:.4f} | val_acc {val_metrics['accuracy']:.4f}"
            scheduler.step(val_auc)
            if np.isfinite(val_auc) and val_auc > best_auc + 1e-4:
                best_auc = val_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
        else:
            scheduler.step(total_acc / total_n)

        print(msg)
        if val_loader is not None and stale_epochs >= patience:
            print(f"early stopping at epoch {ep:02d} | best_val_auc {best_auc:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, norm

@torch.no_grad()
def test_cnn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 512,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)
    y_te = np.asarray(y_test).astype(np.float32)

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    loader = DataLoader(
        SeqDataset(X_te, y_te),
        batch_size=batch_size,
        shuffle=False,
    )

    return eval_cnn_classifier(model, loader, device)

In [9]:
model, norm = train_cnn_on_trainset(
    data_train=data_train,
    y_train=y_train,
    data_val=data_val,
    y_val=y_val,
    T=T,
    epochs=TRAINING_CONFIG["cnn"]["epochs"],
    batch_size=TRAINING_CONFIG["cnn"]["batch_size"],
    lr=TRAINING_CONFIG["cnn"]["lr"],
    use_geom=True,
    add_time=True,
    weight_decay=TRAINING_CONFIG["cnn"]["weight_decay"],
    patience=TRAINING_CONFIG["cnn"]["patience"],
)

metrics, pred_prob_test, pred_label_test, true_label_test = test_cnn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print("TEST metrics:", metrics)
print("pred_prob_test shape:", pred_prob_test.shape)
print("pred_prob_test[:10]:", np.round(pred_prob_test[:10], 4))
print("pred_label_test[:10]:", pred_label_test[:10])
print("true_label_test[:10]:", true_label_test[:10])

KeyboardInterrupt: 

## TCN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score


# =========================================================
# 1) Feature engineering helpers (same as yours)
# =========================================================

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    Invariant under translation/rotation (differences + norms/dots).
    """
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)  # (N,T,9)

def build_features(data: dict, T: int = 100, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    """
    Build X: (N, C, T) from dict.
    NOTE: following your latest choice: NOT using periods_complete, NOT using tAnos_complete.
    """
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)         # (N,T,1)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)         # (N,T,2)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)          # (N,T,3)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)            # (N,T,2)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)          # (N,T,2)
    en = data["energy_complete"][:, sl, :].astype(np.float32)          # (N,T,1)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)     # (N,T,3)

    # Angles -> sin/cos
    ap  = sincos(data["APs_complete"][:, sl, :])                       # (N,T,4)
    lan = sincos(data["LANs_complete"][:, sl, :])                      # (N,T,4)
    inc = sincos(data["incs_complete"][:, sl, :])                      # (N,T,4)

    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)   # (N,T,3,3)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)  # (N,T,3,3)
        geom = pairwise_geom(pos, vel)                                     # (N,T,9)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]      # (1,T,1)
        t = np.repeat(t, N, axis=0)                                        # (N,T,1)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)  # (N,T,C)
    X = X.transpose(0, 2, 1)            # (N,C,T)
    return X.astype(np.float32)


# =========================================================
# 2) Dataset
# =========================================================

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N,C,T)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32).view(-1, 1)

    def __len__(self): 
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


# =========================================================
# 3) TCN Network
# =========================================================

class Chomp1d(nn.Module):
    """Chomp right side to ensure causality (remove future-leaking padding)."""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = int(chomp_size)

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    Residual block:
      (Conv -> BN -> ReLU -> Dropout -> Chomp) x 2 + residual
    """
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_size: int,
        dilation: int,
        dropout: float = 0.2,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        pad = (kernel_size - 1) * dilation

        layers1 = [
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
        ]
        if use_batchnorm:
            layers1.append(nn.BatchNorm1d(out_ch))
        layers1 += [nn.ReLU(), nn.Dropout(dropout), Chomp1d(pad)]

        layers2 = [
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
        ]
        if use_batchnorm:
            layers2.append(nn.BatchNorm1d(out_ch))
        layers2 += [nn.ReLU(), nn.Dropout(dropout), Chomp1d(pad)]

        self.net = nn.Sequential(*(layers1 + layers2))
        self.downsample = None if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.final_relu(out + res)


class TCNBackbone(nn.Module):
    """
    Stack TemporalBlocks with dilations: 1,2,4,8,...
    """
    def __init__(
        self,
        in_ch: int,
        channels: list,
        kernel_size: int = 3,
        dropout: float = 0.2,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        blocks = []
        prev = in_ch
        for i, ch in enumerate(channels):
            blocks.append(
                TemporalBlock(
                    in_ch=prev,
                    out_ch=ch,
                    kernel_size=kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                    use_batchnorm=use_batchnorm,
                )
            )
            prev = ch
        self.network = nn.Sequential(*blocks)

    def forward(self, x):
        return self.network(x)  # (B, C_last, T)


class TCNClassifier(nn.Module):
    """
    Classification head on top of TCN backbone.
    pool="avg": global average pool over time
    pool="last": take last hidden state (causal-friendly)
    """
    def __init__(
        self,
        in_ch: int,
        tcn_channels: list = (128, 256, 256, 256),  # 4 blocks -> receptive field bigger, good for T=100
        kernel_size: int = 3,
        dropout: float = 0.2,
        head_dropout: float = 0.3,
        pool: str = "avg",
        use_batchnorm: bool = True,
    ):
        super().__init__()
        self.backbone = TCNBackbone(
            in_ch=in_ch,
            channels=list(tcn_channels),
            kernel_size=kernel_size,
            dropout=dropout,
            use_batchnorm=use_batchnorm,
        )
        self.pool = pool
        out_ch = int(tcn_channels[-1])

        self.head = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(out_ch, 1),
        )

    def forward(self, x):
        h = self.backbone(x)  # (B, C_last, T)
        if self.pool == "last":
            z = h[:, :, -1]
        else:
            z = h.mean(dim=-1)
        return self.head(z)   # logits


# =========================================================
# 4) Train / Eval utils
# =========================================================

def standardize_train_only(X_tr: np.ndarray, X_te: np.ndarray, eps: float = 1e-6):
    """Standardize each channel using training stats over (N,T)."""
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + eps
    return (X_tr - mu) / sd, (X_te - mu) / sd, (mu, sd)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_prob, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).detach().cpu()
        prob = torch.sigmoid(logits).numpy().ravel()
        all_prob.append(prob)
        all_y.append(yb.numpy().ravel())

    all_prob = np.concatenate(all_prob)
    all_y = np.concatenate(all_y)

    pred = (all_prob >= 0.5).astype(np.int32)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else float("nan")
    f1 = f1_score(all_y, pred) if len(np.unique(all_y)) > 1 else float("nan")
    acc = accuracy_score(all_y, pred)
    return auc, f1, acc


# =========================================================
# 5) Training / Testing (drop-in use)
# =========================================================

def train_tcn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    data_val: dict | None = None,
    y_val: np.ndarray | None = None,
    T: int = 100,
    epochs: int = 40,
    batch_size: int = 128,
    lr: float = 7e-4,
    add_time: bool = True,
    use_geom: bool = True,
    # TCN hyperparams
    tcn_channels=(128, 256, 256, 256),
    kernel_size: int = 3,
    dropout: float = 0.2,
    head_dropout: float = 0.3,
    pool: str = "avg",
    weight_decay: float = 1e-4,
    clip_grad: float | None = 1.0,
    patience: int = 6,
    seed: int = 42,
):
    """
    Returns:
      model, norm_stats (mu, sd)
    """
    # reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Build features
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0]

    # Optional validation
    if (data_val is not None) and (y_val is not None):
        X_va = build_features(data_val, T=T, add_time=add_time, use_geom=use_geom)
        y_va = np.asarray(y_val).astype(np.float32)
        X_tr, X_va, norm = standardize_train_only(X_tr, X_va)
        val_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=512, shuffle=False)
    else:
        # train-only normalization
        mu = X_tr.mean(axis=(0, 2), keepdims=True)
        sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
        X_tr = (X_tr - mu) / sd
        norm = (mu, sd)
        val_loader = None

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = TCNClassifier(
        in_ch=X_tr.shape[1],
        tcn_channels=tcn_channels,
        kernel_size=kernel_size,
        dropout=dropout,
        head_dropout=head_dropout,
        pool=pool,
        use_batchnorm=True,
    ).to(device)

    # imbalance-aware loss
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=lr * 0.1)

    best_auc = -1.0
    best_state = None
    stale_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).view(-1)
            opt.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()

            if clip_grad is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad))

            opt.step()

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            total_n += bs

        msg = f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}"

        # optional validation each epoch
        if val_loader is not None:
            auc, f1, acc = eval_model(model, val_loader, device)
            msg += f" | val_auc {auc:.4f} | val_f1 {f1:.4f} | val_acc {acc:.4f}"

            if np.isfinite(auc) and auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
            scheduler.step(auc if np.isfinite(auc) else 0.0)

        print(msg)
        if val_loader is not None and stale_epochs >= patience:
            print(f"early stopping at epoch {ep:02d} | best_val_auc {best_auc:.4f}")
            break

    # restore best if had validation
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, norm


@torch.no_grad()
def test_tcn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 512,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    loader = DataLoader(SeqDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    auc, f1, acc = eval_model(model, loader, device)
    return auc, f1, acc


# =========================================================
# 6) (Optional) Quick end-to-end usage example
# =========================================================
# model, norm = train_tcn_on_trainset(data_train, y_train, T=100, epochs=20)
# auc, f1, acc = test_tcn(model, data_test, y_test, norm, T=100)
# print("TEST:", auc, f1, acc)

In [ ]:
model, norm = train_tcn_on_trainset(
    data_train=data_train,
    y_train=y_train,  
    data_val=data_val,
    y_val=y_val,
    T=T,
    epochs=TRAINING_CONFIG["tcn"]["epochs"],
    batch_size=TRAINING_CONFIG["tcn"]["batch_size"],
    lr=TRAINING_CONFIG["tcn"]["lr"],
    use_geom=True,
    add_time=True,
    weight_decay=TRAINING_CONFIG["tcn"]["weight_decay"],
    patience=TRAINING_CONFIG["tcn"]["patience"],
)

auc, f1, acc = test_tcn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print("TEST | AUC:", auc, "F1:", f1, "Acc:", acc)


## Attention

In [26]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for (B, T, D)."""
    def __init__(self, d_model: int, max_len: int = 1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len,1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # even
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class AttnClassifier(nn.Module):
    """
    Transformer encoder classifier for inputs (B, C, T).
    - token = one time step
    - output = sequence-level binary logit
    """
    def __init__(
        self,
        in_ch: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_ff: int = 512,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" or "cls"
        max_len: int = 1000,
    ):
        super().__init__()
        assert pooling in ("mean", "cls")

        self.pooling = pooling
        self.in_proj = nn.Linear(in_ch, d_model)

        if pooling == "cls":
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        else:
            self.cls = None

        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B,T,D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.3),
            nn.Linear(d_model, 1),
        )

        # init
        nn.init.xavier_uniform_(self.in_proj.weight)
        nn.init.zeros_(self.in_proj.bias)
        if self.cls is not None:
            nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,T) -> (B,T,C)
        x = x.transpose(1, 2)
        x = self.in_proj(x)  # (B,T,D)

        if self.pooling == "cls":
            B = x.size(0)
            cls_tok = self.cls.expand(B, -1, -1)  # (B,1,D)
            x = torch.cat([cls_tok, x], dim=1)    # (B,T+1,D)

        x = self.pos(x)
        x = self.drop(x)
        x = self.encoder(x)  # (B,T(or T+1),D)

        if self.pooling == "cls":
            seq = x[:, 0, :]          # CLS
        else:
            seq = x.mean(dim=1)       # mean pooling over time

        return self.head(seq)         # (B,1)

def train_attn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    data_val: dict | None = None,
    y_val: np.ndarray | None = None,
    T: int = 100,
    epochs: int = 40,
    batch_size: int = 128,
    lr: float = 3e-4,
    add_time: bool = True,
    use_geom: bool = True,
    d_model: int = 128,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.1,
    pooling: str = "cls",
    use_amp: bool = True,
    weight_decay: float = 1e-4,
    patience: int = 8,
    clip_grad: float = 1.0,
):
    """
    Train on (data_train, y_train) only.
    Returns: model, norm=(mu, sd) computed from train set only.
    """
    # Build features from TRAIN set
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0], "X_tr and y_tr must have same N"

    if (data_val is not None) and (y_val is not None):
        X_va = build_features(data_val, T=T, add_time=add_time, use_geom=use_geom)
        y_va = np.asarray(y_val).astype(np.float32)
        X_tr, X_va, norm = standardize_train_only(X_tr, X_va)
        val_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=512, shuffle=False, drop_last=False)
    else:
        mu = X_tr.mean(axis=(0, 2), keepdims=True)
        sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
        X_tr = (X_tr - mu) / sd
        norm = (mu, sd)
        val_loader = None

    train_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnClassifier(
        in_ch=X_tr.shape[1],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
        max_len=T + (1 if pooling == "cls" else 0) + 5,
    ).to(device)

    # pos_weight from TRAIN only
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=lr * 0.1)

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
    best_auc = -1.0
    best_state = None
    stale_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).view(-1)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits = model(xb).squeeze(-1)
                loss = loss_fn(logits, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad))
            scaler.step(opt)
            scaler.update()

            bs = xb.size(0)
            total_loss += float(loss.detach().cpu().item()) * bs
            total_n += bs

        msg = f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}"
        if val_loader is not None:
            auc, f1, acc = eval_model(model, val_loader, device)
            msg += f" | val_auc {auc:.4f} | val_f1 {f1:.4f} | val_acc {acc:.4f}"
            scheduler.step(auc if np.isfinite(auc) else 0.0)
            if np.isfinite(auc) and auc > best_auc + 1e-4:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
        print(msg)
        if val_loader is not None and stale_epochs >= patience:
            print(f"early stopping at epoch {ep:02d} | best_val_auc {best_auc:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, norm


@torch.no_grad()
def test_attn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 256,
    add_time: bool = True,
    use_geom: bool = True,
):
    """
    Evaluate on external test set (data_test, y_test), using train norm (mu, sd).
    Returns: (auc, f1, acc)
    """
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)
    assert X_te.shape[0] == y_te.shape[0], "X_te and y_te must have same N"

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    test_loader = DataLoader(
        SeqDataset(X_te, y_te),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    auc, f1, acc = eval_model(model, test_loader, device)
    return auc, f1, acc

In [27]:

model, norm = train_attn_on_trainset(
    data_train=data_train,
    y_train=y_train,
    data_val=data_val,
    y_val=y_val,
    T=T,
    epochs=TRAINING_CONFIG["attn"]["epochs"],
    batch_size=TRAINING_CONFIG["attn"]["batch_size"],
    lr=TRAINING_CONFIG["attn"]["lr"],
    use_geom=True,
    add_time=True,
    pooling="cls",
    use_amp=True,
    weight_decay=TRAINING_CONFIG["attn"]["weight_decay"],
    patience=TRAINING_CONFIG["attn"]["patience"],
)


auc, f1, acc = test_attn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print(f"TEST | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")



/tmp/ipykernel_99676/2087478407.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
/tmp/ipykernel_99676/2087478407.py:167: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
/tmp/ipykernel_99676/2087478407.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 01 | train_loss 0.549327 | val_auc 0.9141 | val_f1 0.7583 | val_acc 0.8098


/tmp/ipykernel_99676/2087478407.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 02 | train_loss 0.480245 | val_auc 0.9254 | val_f1 0.7615 | val_acc 0.8117


/tmp/ipykernel_99676/2087478407.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


...

epoch 37 | train_loss 0.215316 | val_auc 0.9689 | val_f1 0.8503 | val_acc 0.9008


/tmp/ipykernel_99676/2087478407.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 38 | train_loss 0.212022 | val_auc 0.9689 | val_f1 0.8517 | val_acc 0.9023


/tmp/ipykernel_99676/2087478407.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 39 | train_loss 0.210683 | val_auc 0.9688 | val_f1 0.8504 | val_acc 0.9013
early stopping at epoch 39 | best_val_auc 0.9691
TEST | AUC 0.9689 | F1 0.8548 | Acc 0.9039


## GNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===========================
# Helpers
# ===========================

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians -> (..., 2k)"""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def build_global_orbital_features(data: dict, T: int) -> np.ndarray:
    """
    Only use the orbital parameters explicitly specified:

      - aratios (2)
      - mratios (1)
      - eccs (2)
      - incs (2) -> sin/cos (4)
      - APs  (2) -> sin/cos (4)
      - LANs (2) -> sin/cos (4)

    Output:
      global_x: (N, T, Fg)

    Total feature dim:
      2 + 1 + 2 + 4 + 4 + 4 = 17
    """
    sl = slice(0, T)

    ar = data["aratios_complete"][:, sl, :].astype(np.float32)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)

    inc = sincos(data["incs_complete"][:, sl, :])
    ap  = sincos(data["APs_complete"][:, sl, :])
    lan = sincos(data["LANs_complete"][:, sl, :])

    g = np.concatenate([ar, mr, ec, inc, ap, lan], axis=-1).astype(np.float32)
    return g

def build_node_features(data: dict, T: int, use_masses: bool = False) -> np.ndarray:
    """
    Node features per time step and body.
    Output: (N, T, 3, Fn)

    If use_masses=False:
      node features = [position xyz, velocity xyz] -> Fn = 6

    If use_masses=True:
      node features = [mass, position xyz, velocity xyz] -> Fn = 7
    """
    sl = slice(0, T)

    pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
    vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)

    if use_masses:
        masses = data["masses_complete"][:, sl, :].astype(np.float32)
        masses = masses[..., None]
        node = np.concatenate([masses, pos, vel], axis=-1).astype(np.float32)
    else:
        node = np.concatenate([pos, vel], axis=-1).astype(np.float32)

    return node

def build_edge_features(data: dict, T: int, eps: float = 1e-8):
    """
    Build directed edge features for the 3-body fully connected graph (no self loops).

    Return:
      edge_feat: (N, T, E=6, Fe)
      senders: np.ndarray shape (6,)
      receivers: np.ndarray shape (6,)

    Edge features:
      - relative position dx,dy,dz (3)
      - distance ||d|| (1)
      - relative velocity dvx,dvy,dvz (3)
      - relative speed ||dv|| (1)
      - radial speed (1)
    total Fe = 9
    """
    sl = slice(0, T)
    pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
    vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)

    senders = np.array([0, 0, 1, 1, 2, 2], dtype=np.int64)
    receivers = np.array([1, 2, 0, 2, 0, 1], dtype=np.int64)
    E = len(senders)

    edge_feats = []
    for e in range(E):
        i = senders[e]
        j = receivers[e]

        d = pos[:, :, j, :] - pos[:, :, i, :]
        u = vel[:, :, j, :] - vel[:, :, i, :]

        r = np.linalg.norm(d, axis=-1, keepdims=True)
        v = np.linalg.norm(u, axis=-1, keepdims=True)
        rdot = (np.sum(d * u, axis=-1, keepdims=True)) / (r + eps)

        ef = np.concatenate([d, r, u, v, rdot], axis=-1).astype(np.float32)
        edge_feats.append(ef)

    edge_feat = np.stack(edge_feats, axis=2)
    return edge_feat, senders, receivers

def build_graph_data(data: dict, T: int = 100, use_masses: bool = False):
    node_x = build_node_features(data, T, use_masses=use_masses)
    edge_x, senders, receivers = build_edge_features(data, T)
    global_x = build_global_orbital_features(data, T)
    return node_x, edge_x, global_x, senders, receivers

# ===========================
# Dataset
# ===========================

class GraphSeqDataset(Dataset):
    def __init__(self, node_x, edge_x, global_x, y):
        self.node_x = torch.tensor(node_x, dtype=torch.float32)
        self.edge_x = torch.tensor(edge_x, dtype=torch.float32)
        self.global_x = torch.tensor(global_x, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32)

    def __len__(self):
        return self.node_x.shape[0]

    def __getitem__(self, i):
        return self.node_x[i], self.edge_x[i], self.global_x[i], self.y[i]

# ===========================
# Graph model
# ===========================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

class GraphEncoderStep(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim=128, msg_dim=128, out_dim=128):
        super().__init__()
        self.node_proj = nn.Linear(node_dim + global_dim, hidden_dim)
        self.edge_mlp = MLP(2 * hidden_dim + edge_dim + global_dim, msg_dim, msg_dim)
        self.node_update = MLP(hidden_dim + msg_dim + global_dim, hidden_dim, hidden_dim)
        self.readout = MLP(hidden_dim + global_dim, hidden_dim, out_dim)

    def forward(self, node_x, edge_x, global_x, senders, receivers):
        B, Nn, _ = node_x.shape
        E = edge_x.shape[1]

        g_node = global_x.unsqueeze(1).expand(B, Nn, global_x.shape[-1])
        h = self.node_proj(torch.cat([node_x, g_node], dim=-1))

        h_send = h[:, senders, :]
        h_recv = h[:, receivers, :]
        g_edge = global_x.unsqueeze(1).expand(B, E, global_x.shape[-1])

        m_in = torch.cat([h_send, h_recv, edge_x, g_edge], dim=-1)
        m = self.edge_mlp(m_in)

        agg = torch.zeros(B, Nn, m.shape[-1], device=node_x.device, dtype=node_x.dtype)
        for e, r in enumerate(receivers):
            agg[:, r, :] += m[:, e, :]

        node_upd_in = torch.cat([h, agg, g_node], dim=-1)
        h = self.node_update(node_upd_in)

        h_pool = h.mean(dim=1)
        g_t = self.readout(torch.cat([h_pool, global_x], dim=-1))
        return g_t

class GraphGRUClassifier(nn.Module):
    """
    For each time step:
      G_t -> GraphEncoder -> g_t

    Then:
      (g_1,...,g_T) -> GRU -> z -> MLP -> binary logit
    """
    def __init__(
        self,
        node_dim,
        edge_dim,
        global_dim,
        graph_hidden=128,
        msg_dim=128,
        graph_out_dim=128,
        gru_hidden=128,
        num_gru_layers=1,
        dropout=0.3,
    ):
        super().__init__()
        self.graph_encoder = GraphEncoderStep(
            node_dim=node_dim,
            edge_dim=edge_dim,
            global_dim=global_dim,
            hidden_dim=graph_hidden,
            msg_dim=msg_dim,
            out_dim=graph_out_dim,
        )

        self.temporal = nn.GRU(
            input_size=graph_out_dim,
            hidden_size=gru_hidden,
            num_layers=num_gru_layers,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.Linear(gru_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, node_x, edge_x, global_x, senders, receivers):
        B, T, _, _ = node_x.shape

        g_seq = []
        for t in range(T):
            g_t = self.graph_encoder(
                node_x[:, t],
                edge_x[:, t],
                global_x[:, t],
                senders,
                receivers,
            )
            g_seq.append(g_t)

        g_seq = torch.stack(g_seq, dim=1)
        _, h_n = self.temporal(g_seq)
        z = h_n[-1]
        return self.head(z)

# ===========================
# Training / Eval
# ===========================

@torch.no_grad()
def eval_graph_model(model, loader, device, senders, receivers):
    model.eval()
    all_prob, all_target = [], []

    senders_t = torch.tensor(senders, dtype=torch.long, device=device)
    receivers_t = torch.tensor(receivers, dtype=torch.long, device=device)

    for node_x, edge_x, global_x, yb in loader:
        node_x = node_x.to(device)
        edge_x = edge_x.to(device)
        global_x = global_x.to(device)

        logits = model(node_x, edge_x, global_x, senders_t, receivers_t).squeeze(-1)
        prob = torch.sigmoid(logits)

        all_prob.append(prob.detach().cpu().numpy())
        all_target.append(yb.numpy())

    all_prob = np.concatenate(all_prob, axis=0)
    all_target = np.concatenate(all_target, axis=0)
    pred = (all_prob >= 0.5).astype(np.int32)

    metrics = {
        "auc": float(roc_auc_score(all_target, all_prob)),
        "f1": float(f1_score(all_target, pred)),
        "accuracy": float(accuracy_score(all_target, pred)),
    }
    return metrics, all_prob, pred, all_target

def train_graph_model_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    data_val: dict | None = None,
    y_val: np.ndarray | None = None,
    T: int = 100,
    epochs: int = 30,
    batch_size: int = 192,
    lr: float = 5e-4,
    use_masses: bool = False,
    weight_decay: float = 1e-4,
    patience: int = 6,
):
    node_tr, edge_tr, global_tr, senders, receivers = build_graph_data(
        data_train, T=T, use_masses=use_masses
    )
    y_tr = np.asarray(y_train).astype(np.float32)

    node_mu = node_tr.mean(axis=(0, 1, 2), keepdims=True)
    node_sd = node_tr.std(axis=(0, 1, 2), keepdims=True) + 1e-6
    edge_mu = edge_tr.mean(axis=(0, 1, 2), keepdims=True)
    edge_sd = edge_tr.std(axis=(0, 1, 2), keepdims=True) + 1e-6
    global_mu = global_tr.mean(axis=(0, 1), keepdims=True)
    global_sd = global_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    node_tr = (node_tr - node_mu) / node_sd
    edge_tr = (edge_tr - edge_mu) / edge_sd
    global_tr = (global_tr - global_mu) / global_sd

    norm = {
        "node_mu": node_mu, "node_sd": node_sd,
        "edge_mu": edge_mu, "edge_sd": edge_sd,
        "global_mu": global_mu, "global_sd": global_sd,
    }

    if (data_val is not None) and (y_val is not None):
        node_va, edge_va, global_va, _, _ = build_graph_data(data_val, T=T, use_masses=use_masses)
        y_va = np.asarray(y_val).astype(np.float32)
        node_va = (node_va - node_mu) / node_sd
        edge_va = (edge_va - edge_mu) / edge_sd
        global_va = (global_va - global_mu) / global_sd
        val_loader = DataLoader(GraphSeqDataset(node_va, edge_va, global_va, y_va), batch_size=384, shuffle=False, drop_last=False)
    else:
        val_loader = None

    train_loader = DataLoader(
        GraphSeqDataset(node_tr, edge_tr, global_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GraphGRUClassifier(
        node_dim=node_tr.shape[-1],
        edge_dim=edge_tr.shape[-1],
        global_dim=global_tr.shape[-1],
        graph_hidden=128,
        msg_dim=128,
        graph_out_dim=128,
        gru_hidden=128,
        num_gru_layers=1,
        dropout=0.3,
    ).to(device)

    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=lr * 0.1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_auc = -1.0
    best_state = None
    stale_epochs = 0

    senders_t = torch.tensor(senders, dtype=torch.long, device=device)
    receivers_t = torch.tensor(receivers, dtype=torch.long, device=device)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_acc = 0.0
        total_n = 0

        for node_x, edge_x, global_x, yb in train_loader:
            node_x = node_x.to(device)
            edge_x = edge_x.to(device)
            global_x = global_x.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(node_x, edge_x, global_x, senders_t, receivers_t).squeeze(-1)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()

            bs = node_x.size(0)
            total_loss += loss.item() * bs
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_acc += float((pred == yb).float().mean().item()) * bs
            total_n += bs

        msg = f"epoch {ep:02d} | train_loss {total_loss / total_n:.6f} | train_acc {total_acc / total_n:.4f}"
        if val_loader is not None:
            val_metrics, _, _, _ = eval_graph_model(model, val_loader, device, senders, receivers)
            val_auc = val_metrics["auc"]
            msg += f" | val_auc {val_auc:.4f} | val_f1 {val_metrics['f1']:.4f} | val_acc {val_metrics['accuracy']:.4f}"
            scheduler.step(val_auc)
            if np.isfinite(val_auc) and val_auc > best_auc + 1e-4:
                best_auc = val_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
        print(msg)
        if val_loader is not None and stale_epochs >= patience:
            print(f"early stopping at epoch {ep:02d} | best_val_auc {best_auc:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, norm, senders, receivers

@torch.no_grad()
def test_graph_model(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    senders,
    receivers,
    T: int = 100,
    batch_size: int = 512,
    use_masses: bool = False,
):
    node_te, edge_te, global_te, _, _ = build_graph_data(
        data_test, T=T, use_masses=use_masses
    )
    y_te = np.asarray(y_test).astype(np.float32)

    node_te = (node_te - norm["node_mu"]) / norm["node_sd"]
    edge_te = (edge_te - norm["edge_mu"]) / norm["edge_sd"]
    global_te = (global_te - norm["global_mu"]) / norm["global_sd"]

    device = next(model.parameters()).device
    loader = DataLoader(
        GraphSeqDataset(node_te, edge_te, global_te, y_te),
        batch_size=batch_size,
        shuffle=False
    )

    return eval_graph_model(model, loader, device, senders, receivers)



In [ ]:
model, norm, senders, receivers = train_graph_model_on_trainset(
    data_train,
    y_train,
    data_val,
    y_val,
    T=100,
    epochs=TRAINING_CONFIG["gnn"]["epochs"],
    batch_size=TRAINING_CONFIG["gnn"]["batch_size"],
    lr=TRAINING_CONFIG["gnn"]["lr"],
    use_masses=False,
    weight_decay=TRAINING_CONFIG["gnn"]["weight_decay"],
    patience=TRAINING_CONFIG["gnn"]["patience"],
)

metrics, pred_prob_test, pred_label_test, true_label_test = test_graph_model(
    model,
    data_test,
    y_test,
    norm,
    senders,
    receivers,
    T=100,
    batch_size=512,
    use_masses=False,
)



In [ ]:
print(metrics)
print("pred_prob_test shape:", pred_prob_test.shape)
print("pred_prob_test[:10]:", np.round(pred_prob_test[:10], 4))
print("pred_label_test[:10]:", pred_label_test[:10])
print("true_label_test[:10]:", true_label_test[:10])



## Initial Conditions + Prefix

- New case: use two separate heads instead of concatenating raw inputs.
- `initial conditions` go through a static head, while the early trajectory prefix goes through a temporal head, and the fused representation predicts the final label.

In [ ]:
import math

def build_initial_condition_features(initial_data: dict) -> np.ndarray:
    def sincos_static(x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

    ar = initial_data["aratios"].astype(np.float32)
    mr = initial_data["mratios"].astype(np.float32)
    ec = initial_data["eccs"].astype(np.float32)
    inc = sincos_static(initial_data["incs"])
    ap = sincos_static(initial_data["APs"])
    lan = sincos_static(initial_data["LANs"])
    return np.concatenate([ar, mr, ec, inc, ap, lan], axis=-1).astype(np.float32)


def standardize_static_train_only(X_tr: np.ndarray, X_other: np.ndarray, eps: float = 1e-6):
    mu = X_tr.mean(axis=0, keepdims=True)
    sd = X_tr.std(axis=0, keepdims=True) + eps
    return (X_tr - mu) / sd, (X_other - mu) / sd, (mu, sd)

class ICPrefixDataset(Dataset):
    def __init__(self, x_ic: np.ndarray, x_prefix: np.ndarray, y: np.ndarray):
        self.x_ic = torch.tensor(x_ic, dtype=torch.float32)
        self.x_prefix = torch.tensor(x_prefix, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.x_ic[idx], self.x_prefix[idx], self.y[idx]

class StaticICHead(nn.Module):
    def __init__(self, ic_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ic_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x_ic: torch.Tensor) -> torch.Tensor:
        return self.net(x_ic)

class CNNPrefixHead(nn.Module):
    def __init__(self, prefix_ch: int, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(prefix_ch, 96, kernel_size=7, padding=3),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Conv1d(96, out_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )

    def forward(self, x_prefix: torch.Tensor) -> torch.Tensor:
        return self.net(x_prefix)

class TCNResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        padding = dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        x = self.dropout(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout(self.bn2(self.conv2(x)))
        return torch.relu(x + residual)

class TCNPrefixHead(nn.Module):
    def __init__(self, prefix_ch: int, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            TCNResidualBlock(prefix_ch, 96, dilation=1),
            TCNResidualBlock(96, out_dim, dilation=2),
            TCNResidualBlock(out_dim, out_dim, dilation=4),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )

    def forward(self, x_prefix: torch.Tensor) -> torch.Tensor:
        return self.net(x_prefix)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]

class AttentionPrefixHead(nn.Module):
    def __init__(self, prefix_ch: int, d_model: int = 128, out_dim: int = 128, nhead: int = 4, num_layers: int = 2):
        super().__init__()
        self.in_proj = nn.Linear(prefix_ch, d_model)
        self.pos = PositionalEncoding(d_model=d_model, max_len=2048)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.out_proj = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, out_dim),
            nn.ReLU(),
        )

    def forward(self, x_prefix: torch.Tensor) -> torch.Tensor:
        x = x_prefix.transpose(1, 2)
        x = self.in_proj(x)
        x = self.pos(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.out_proj(x)

class ICPrefixTwoHeadClassifier(nn.Module):
    def __init__(self, ic_dim: int, prefix_ch: int, prefix_encoder: str = "cnn"):
        super().__init__()
        self.ic_head = StaticICHead(ic_dim=ic_dim, hidden_dim=64)
        if prefix_encoder == "cnn":
            self.prefix_head = CNNPrefixHead(prefix_ch=prefix_ch, out_dim=128)
        elif prefix_encoder == "tcn":
            self.prefix_head = TCNPrefixHead(prefix_ch=prefix_ch, out_dim=128)
        elif prefix_encoder == "attn":
            self.prefix_head = AttentionPrefixHead(prefix_ch=prefix_ch, d_model=128, out_dim=128, nhead=4, num_layers=2)
        else:
            raise ValueError(f"Unknown prefix_encoder: {prefix_encoder}")
        self.fusion = nn.Sequential(
            nn.Linear(64 + 128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x_ic: torch.Tensor, x_prefix: torch.Tensor) -> torch.Tensor:
        h_ic = self.ic_head(x_ic)
        h_prefix = self.prefix_head(x_prefix)
        return self.fusion(torch.cat([h_ic, h_prefix], dim=-1)).squeeze(-1)

@torch.no_grad()
def eval_ic_prefix_two_head(model, loader, device):
    model.eval()
    all_prob, all_target = [], []
    for x_ic, x_prefix, yb in loader:
        x_ic = x_ic.to(device)
        x_prefix = x_prefix.to(device)
        logits = model(x_ic, x_prefix)
        prob = torch.sigmoid(logits)
        all_prob.append(prob.detach().cpu().numpy())
        all_target.append(yb.numpy())
    all_prob = np.concatenate(all_prob, axis=0)
    all_target = np.concatenate(all_target, axis=0)
    pred = (all_prob >= 0.5).astype(np.int32)
    metrics = {
        "auc": float(roc_auc_score(all_target, all_prob)),
        "f1": float(f1_score(all_target, pred)),
        "accuracy": float(accuracy_score(all_target, pred)),
    }
    return metrics, all_prob, pred, all_target

def train_ic_prefix_two_head(
    x_ic_train: np.ndarray,
    x_prefix_train: np.ndarray,
    y_train: np.ndarray,
    x_ic_val: np.ndarray,
    x_prefix_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 40,
    batch_size: int = 256,
    lr: float = 5e-4,
    weight_decay: float = 1e-4,
    patience: int = 6,
    prefix_encoder: str = "cnn",
):
    train_loader = DataLoader(ICPrefixDataset(x_ic_train, x_prefix_train, y_train), batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(ICPrefixDataset(x_ic_val, x_prefix_val, y_val), batch_size=512, shuffle=False, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ICPrefixTwoHeadClassifier(ic_dim=x_ic_train.shape[1], prefix_ch=x_prefix_train.shape[1], prefix_encoder=prefix_encoder).to(device)

    pos = float(y_train.sum())
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_auc = -1.0
    best_state = None
    stale_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0
        for x_ic, x_prefix, yb in train_loader:
            x_ic = x_ic.to(device)
            x_prefix = x_prefix.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(x_ic, x_prefix)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            bs = x_ic.shape[0]
            total_loss += float(loss.detach().cpu().item()) * bs
            total_n += bs

        val_metrics, _, _, _ = eval_ic_prefix_two_head(model, val_loader, device)
        msg = f"epoch {ep:02d} | train_loss {total_loss / total_n:.6f} | val_auc {val_metrics['auc']:.4f} | val_f1 {val_metrics['f1']:.4f} | val_acc {val_metrics['accuracy']:.4f}"
        print(msg)

        if np.isfinite(val_metrics['auc']) and val_metrics['auc'] > best_auc + 1e-4:
            best_auc = val_metrics['auc']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= patience:
            print(f"early stopping at epoch {ep:02d} | best_val_auc {best_auc:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, device


In [ ]:
X_prefix_train = build_features(data_train, T=T, add_time=True, use_geom=True)
X_prefix_val   = build_features(data_val,   T=T, add_time=True, use_geom=True)
X_prefix_test  = build_features(data_test,  T=T, add_time=True, use_geom=True)

X_ic_train = build_initial_condition_features(initial_train)
X_ic_val   = build_initial_condition_features(initial_val)
X_ic_test  = build_initial_condition_features(initial_test)

X_prefix_train, X_prefix_val, (prefix_mu, prefix_sd) = standardize_train_only(X_prefix_train, X_prefix_val)
X_prefix_test = (X_prefix_test - prefix_mu) / prefix_sd
X_ic_train, X_ic_val, (ic_mu, ic_sd) = standardize_static_train_only(X_ic_train, X_ic_val)
X_ic_test = (X_ic_test - ic_mu) / ic_sd

print("IC feature dim:", X_ic_train.shape[1])
print("prefix shape:", X_prefix_train.shape, "finite:", np.isfinite(X_prefix_train).all())
print("ic shape:", X_ic_train.shape, "finite:", np.isfinite(X_ic_train).all())

IC_PREFIX_MODEL_SPECS = {
    "IC-MLP + Prefix-CNN": ("cnn", TRAINING_CONFIG["cnn"]),
    "IC-MLP + Prefix-TCN": ("tcn", TRAINING_CONFIG["tcn"]),
    "IC-MLP + Prefix-Attention": ("attn", TRAINING_CONFIG["attn"]),
}

ic_prefix_runs = {}
for model_name, (prefix_encoder, cfg) in IC_PREFIX_MODEL_SPECS.items():
    print(f"\n=== {model_name} ===")
    model_icp, device_icp = train_ic_prefix_two_head(
        X_ic_train,
        X_prefix_train,
        y_train,
        X_ic_val,
        X_prefix_val,
        y_val,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        patience=cfg["patience"],
        prefix_encoder=prefix_encoder,
    )

    train_loader_icp = DataLoader(ICPrefixDataset(X_ic_train, X_prefix_train, y_train), batch_size=512, shuffle=False)
    test_loader_icp  = DataLoader(ICPrefixDataset(X_ic_test,  X_prefix_test,  y_test),  batch_size=512, shuffle=False)

    train_metrics_icp, _, _, _ = eval_ic_prefix_two_head(model_icp, train_loader_icp, device_icp)
    test_metrics_icp, pred_prob_icp, pred_label_icp, true_label_icp = eval_ic_prefix_two_head(model_icp, test_loader_icp, device_icp)
    ic_prefix_runs[model_name] = {
        "model": model_icp,
        "device": device_icp,
        "train_metrics": train_metrics_icp,
        "test_metrics": test_metrics_icp,
        "pred_prob_test": pred_prob_icp,
        "pred_label_test": pred_label_icp,
        "true_label_test": true_label_icp,
    }
    print("train:", train_metrics_icp)
    print("test :", test_metrics_icp)
    print("pred_prob_test[:10]:", np.round(pred_prob_icp[:10], 4))
    print("pred_label_test[:10]:", pred_label_icp[:10])
    print("true_label_test[:10]:", true_label_icp[:10])
